# Dataset · Graphistry

Run All prepares graphs and executes GFQL locally. Hosted rendering stays disabled. [Graphistry reference](https://pygraphistry.readthedocs.io/en/latest/cheatsheet.html).

In [ ]:
import json
import sys
import unicodedata
from pathlib import Path

import duckdb
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "scripts/plazapd/mappedin.py").is_file()
             and (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the repository or nbs/plazapd.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.plazapd.mappedin import load_tables

DATA_DIR = ROOT / "data/plazapd"
tables = load_tables(DATA_DIR, format="parquet")
manifest = json.loads((DATA_DIR / "manifest.json").read_text())


def text_key(value):
    return "".join(c for c in unicodedata.normalize("NFKD", value).casefold() if c.isalnum())


venue_aliases = {text_key(value) for value in [
    manifest.get("venue_name"), manifest.get("venue_id"), *tables["venues"].name.dropna()
] if isinstance(value, str) and value}


def venue_label(value):
    if isinstance(value, str) and any(alias in text_key(value) for alias in venue_aliases):
        return "Venue"
    return value


for table in tables.values():
    for column in ("name", "description"):
        if column in table:
            table[column] = table[column].map(venue_label).astype(table[column].dtype)

assert tables["entities"].entity_id.is_unique
assert tables["relationships"][["src", "dst"]].isin(set(tables["entities"].entity_id)).all().all()
assert tables["distances"].loc[tables["distances"].status.eq("unreachable"), "distance_m"].isna().all()
pd.options.display.max_colwidth = 50
pd.options.display.max_rows = 25

db = duckdb.connect(":memory:")
for name, frame in tables.items():
    db.register(name, frame)


def sql(query, parameters=None):
    return db.execute(query, parameters).df()

In [ ]:
_ = db.execute("""
CREATE OR REPLACE TEMP VIEW location_spaces AS
SELECT DISTINCT r.src AS location_id, r.dst AS space_id
FROM relationships r JOIN locations l ON l.entity_id = r.src
JOIN spaces s ON s.entity_id = r.dst WHERE r.relation = 'spaces';

CREATE OR REPLACE TEMP VIEW space_floors AS
SELECT DISTINCT r.src AS space_id, r.dst AS floor_entity_id
FROM relationships r JOIN spaces s ON s.entity_id = r.src
JOIN floors f ON f.entity_id = r.dst WHERE r.relation = 'floor';

CREATE OR REPLACE TEMP VIEW location_floors AS
SELECT DISTINCT l.location_id, f.floor_entity_id
FROM location_spaces l JOIN space_floors f USING (space_id);

CREATE OR REPLACE TEMP VIEW location_category_links AS
SELECT DISTINCT r.src AS location_id, r.dst AS category_id
FROM relationships r JOIN locations l ON l.entity_id = r.src
JOIN categories c ON c.entity_id = r.dst WHERE r.relation = 'categories';

CREATE OR REPLACE TEMP VIEW node_catalog AS
SELECT e.entity_id, e.id, coalesce(nullif(trim(e.name), ''), e.entity_id) AS label,
       CASE WHEN s.entity_id IS NOT NULL THEN 'store'
            WHEN a.entity_id IS NOT NULL THEN 'amenity'
            WHEN e.collection = 'spaces' THEN 'space'
            WHEN e.collection = 'floors' THEN 'floor'
            WHEN e.collection = 'categories' THEN 'category'
            WHEN e.collection = 'nodes' THEN 'navigation'
            ELSE e.type END AS kind,
       e.hidden, e.navigable, e.amenity
FROM entities e LEFT JOIN stores s USING (entity_id) LEFT JOIN amenities a USING (entity_id);
""")

In [ ]:
nodes = sql("SELECT * FROM node_catalog ORDER BY kind, label, entity_id")
relations = tables["relationships"].copy()
location_spaces = sql("SELECT * FROM location_spaces")
space_floors = sql("SELECT * FROM space_floors")
location_floors = sql("SELECT * FROM location_floors")
location_category_links = sql("SELECT * FROM location_category_links")
floor_names = tables["floors"].set_index("entity_id").name
location_floor_names = location_floors.assign(
    floor=location_floors.floor_entity_id.map(floor_names)
).groupby("location_id").floor.agg(lambda v: " | ".join(sorted(set(v))))
nodes["floors"] = nodes.entity_id.map(location_floor_names).fillna("")
for column in ["hidden", "navigable"]:
    nodes[column] = nodes[column].astype("string").fillna("Unknown")
nodes["amenity"] = nodes.amenity.fillna("")


def membership_graph(store_name="", floor=None, include_amenities=True):
    selected = set(nodes.loc[nodes.kind.eq("store") & nodes.label.str.contains(
        store_name, case=False, regex=False), "entity_id"])
    if include_amenities:
        selected |= set(nodes.loc[nodes.kind.eq("amenity"), "entity_id"])
    selected_floors = set(tables["floors"].entity_id)
    if floor is not None:
        floor_rows = tables["floors"]
        selected_floors = set(floor_rows.loc[
            floor_rows.entity_id.eq(floor) | floor_rows.name.eq(floor), "entity_id"])
        if not selected_floors:
            raise ValueError("Unknown floor; use an exact name or entity ID from the floor table.")
        selected &= set(location_floors.loc[
            location_floors.floor_entity_id.isin(selected_floors), "location_id"])
    selected_spaces = set(location_spaces.loc[location_spaces.location_id.isin(selected), "space_id"])
    if floor is not None:
        selected_spaces &= set(space_floors.loc[
            space_floors.floor_entity_id.isin(selected_floors), "space_id"])
    selected |= selected_spaces | set(space_floors.loc[
        space_floors.space_id.isin(selected_spaces), "floor_entity_id"])
    selected |= set(location_category_links.loc[location_category_links.location_id.isin(selected), "category_id"])
    ns = nodes.loc[nodes.entity_id.isin(selected)].copy().reset_index(drop=True)
    es = relations.loc[relations.src.isin(selected) & relations.dst.isin(selected)].copy().reset_index(drop=True)
    return ns, es


def walking_graph(origin=None, k_stores=5, k_amenities=3):
    if any(not isinstance(k, int) or isinstance(k, bool) or k < 0 for k in (k_stores, k_amenities)):
        raise ValueError("Neighbour counts must be nonnegative integers.")
    candidates = tables["stores"].loc[tables["stores"].navigable.fillna(False)].sort_values(["name", "entity_id"])
    if origin is not None:
        candidates = candidates.loc[candidates.entity_id.eq(origin) | candidates.name.eq(origin)]
    if candidates.empty:
        raise ValueError("Choose a navigable store by exact name or entity ID.")
    source = candidates.iloc[0]
    routes = tables["distances"]
    reachable = routes.loc[routes.from_id.eq(source.id) & routes.status.eq("reachable")].copy()
    selected = []
    for table, count, kind in [("stores", k_stores, "store"), ("amenities", k_amenities, "amenity")]:
        ranked = reachable.loc[reachable.to_id.isin(tables[table].id)].sort_values(["distance_m", "to_id"]).head(count)
        targets = ranked.merge(tables[table][["id", "entity_id"]], left_on="to_id", right_on="id")
        selected.extend({"src": source.entity_id, "dst": row.entity_id,
                         "relation": "nearest_reachable_" + kind, "distance_m": row.distance_m}
                        for row in targets.itertuples(index=False))
    es = pd.DataFrame(selected, columns=["src", "dst", "relation", "distance_m"])
    ids = {source.entity_id} | set(es.dst)
    ns = nodes.loc[nodes.entity_id.isin(ids)].copy().reset_index(drop=True)
    return source.entity_id, ns, es

## Full source graph

In [ ]:
import graphistry
from graphistry import e_forward, e_reverse, n

COLORS = {"category":"#9b59b6","store":"#2874a6","amenity":"#148f77","space":"#d68910","floor":"#c0392b","navigation":"#8c8c8c"}


def bind_graph(ns, es):
    return graphistry.nodes(ns,"entity_id").edges(es,"src","dst").bind(
        point_title="label",point_label="label",edge_title="relation"
    ).encode_point_color("kind",categorical_mapping=COLORS,default_mapping="#777777")


full_graph = bind_graph(nodes,relations)
print(f"{len(nodes):,} nodes · {len(relations):,} directed source edges")
display(nodes.groupby("kind").size().rename("nodes").to_frame())
display(relations.groupby("relation").size().rename("edges").to_frame())

## Store → space → floor

In [ ]:
store_floor_graph = full_graph.gfql([
    n({"kind":"store"}),e_forward({"relation":"spaces"}),n({"kind":"space"}),
    e_forward({"relation":"floor"}),n({"kind":"floor"}),
],engine="pandas")
path_edges = store_floor_graph._edges[["src","relation","dst"]].copy()
labels = nodes.set_index("entity_id").label
path_edges["source"] = path_edges.src.map(labels)
path_edges["target"] = path_edges.dst.map(labels)
print(f"{len(store_floor_graph._nodes):,} nodes · {len(path_edges):,} edges")
display(path_edges[["source","relation","target"]].head(15))

## Stores sharing a category

`ORIGIN` accepts an exact navigable-store name or entity ID; ties use entity ID.

In [ ]:
ORIGIN = None
K_STORES = 5
K_AMENITIES = 3

origin_id, walking_nodes, walking_edges = walking_graph(ORIGIN,K_STORES,K_AMENITIES)
category_graph = full_graph.gfql([
    n({"entity_id":origin_id}),e_forward({"relation":"categories"}),n({"kind":"category"}),
    e_reverse({"relation":"categories"}),n({"kind":"store"}),
],engine="pandas")
display(category_graph._nodes.loc[category_graph._nodes.kind.eq("store"),["entity_id","label"]]
        .sort_values(["label","entity_id"]).head(15))
print(f"{category_graph._nodes.kind.eq('store').sum():,} stores in matching paths (including origin when matched)")

## Membership and walking graphs

Membership uses explicit links; nearest-neighbour edges are derived from directed walking distances in metres.

In [ ]:
STORE_NAME = ""
FLOOR = "Piso 1"
INCLUDE_AMENITIES = True

membership_nodes,membership_edges = membership_graph(STORE_NAME,FLOOR,INCLUDE_AMENITIES)
membership = bind_graph(membership_nodes,membership_edges)
walking = bind_graph(walking_nodes,walking_edges)
walking_table = walking_edges.assign(destination=walking_edges.dst.map(labels))
display(walking_table[["destination","relation","distance_m"]].round({"distance_m":2}))
graphs = {"membership":membership,"walking":walking,"store_floor":store_floor_graph,
          "shared_category":category_graph,"full":full_graph}
display(pd.DataFrame([{"graph":key,"nodes":len(g._nodes),"edges":len(g._edges)} for key,g in graphs.items()]))

## Hosted rendering · disabled by default

Set `GRAPHISTRY_SERVER` (hostname), `GRAPHISTRY_PERSONAL_KEY_ID`, and `GRAPHISTRY_PERSONAL_KEY_SECRET` in the kernel environment; enable the switch to upload `VIEW`. Credentials stay outside the notebook. [Graphistry authentication](https://pygraphistry.readthedocs.io/en/latest/cheatsheet.html).

In [ ]:
UPLOAD_TO_GRAPHISTRY = False
VIEW = "membership"

if UPLOAD_TO_GRAPHISTRY:
    import os

    if VIEW not in graphs:
        raise ValueError(f"Choose VIEW from {list(graphs)}")
    required = ["GRAPHISTRY_SERVER","GRAPHISTRY_PERSONAL_KEY_ID","GRAPHISTRY_PERSONAL_KEY_SECRET"]
    if not all(os.environ.get(key) for key in required):
        raise ValueError("Set the Graphistry server and personal-key environment variables above.")
    selected_graph = graphs[VIEW]
    if selected_graph._nodes.empty:
        print("No selected nodes to render.")
    else:
        graphistry.register(api=3,protocol="https",server=os.environ["GRAPHISTRY_SERVER"],
                            personal_key_id=os.environ["GRAPHISTRY_PERSONAL_KEY_ID"],
                            personal_key_secret=os.environ["GRAPHISTRY_PERSONAL_KEY_SECRET"])
        display(selected_graph.plot(name=f"Dataset · {VIEW}",render="ipython"))
else:
    print("Graph prepared; hosted rendering disabled.")